In [1]:
import pandas as pd
import json
import numpy as np
import os
from glob import glob
os.chdir('..')

In [43]:
from src.utils.eval_utils import calculate_metrics

In [44]:
seed = '123'
dataset_type = 'hoasa_hotel'
lang = 'indo'
results_paths = glob(f'outputs/evals/{dataset_type}/{lang}/*/seed_{seed}/*/*/*/*/inference_results.json')

In [46]:
results_dict = {}
for path in results_paths:
	with open(path, 'r') as f:
		results = json.load(f)
	dataset_folder = path.split('/')[4]
	decoding = path.split('/')[-2]
	results_dict[f'{dataset_folder}-{decoding}'] = results

In [60]:
df_dict = {}
for key, results in results_dict.items():
	df_dict[key] = pd.DataFrame(results)
	df_dict[key]['metrics'] = df_dict[key].apply(lambda row: calculate_metrics([row['target_list']], [row['prediction_list']], task='aos'), axis=1)
	df_dict[key]['f1_score'] = df_dict[key]['metrics'].apply(lambda x: x['f1_aos'])
	df_dict[key]['target_list'] = df_dict[key]['target_list'].apply(lambda x: '\n--------\n'.join(x))
	df_dict[key]['prediction_list'] = df_dict[key]['prediction_list'].apply(lambda x: '\n--------\n'.join(x))

In [61]:
df_dict.keys()

dict_keys(['indolegoabsa_multitask_old-constrained_decoding', 'indolegoabsa_multitask_old-unconstrained_decoding', 'gas-constrained_decoding', 'gas-unconstrained_decoding', 'legoabsa_multitask_old-constrained_decoding', 'legoabsa_multitask_old-unconstrained_decoding', 'mvp_aos_augment-constrained_decoding', 'mvp_aos_augment-unconstrained_decoding', 'mvp_aos-constrained_decoding', 'mvp_aos-unconstrained_decoding', 'legoabsa_tasktransfer_old-constrained_decoding', 'legoabsa_tasktransfer_old-unconstrained_decoding'])

In [62]:
new_df = df_dict['mvp_aos_augment-constrained_decoding'].copy()
for key, df in df_dict.items():
	if key == 'mvp_aos_augment-constrained_decoding':
		continue
	dataset_folder = key.split('-')[0]
	decoding = key.split('-')[1]

	key = key.replace('_old', '')

	new_df = new_df.merge(df, on='sentence_id', suffixes=('', f'-{key}')).copy()
	

In [63]:
list(df_dict.keys())

['indolegoabsa_multitask_old-constrained_decoding',
 'indolegoabsa_multitask_old-unconstrained_decoding',
 'gas-constrained_decoding',
 'gas-unconstrained_decoding',
 'legoabsa_multitask_old-constrained_decoding',
 'legoabsa_multitask_old-unconstrained_decoding',
 'mvp_aos_augment-constrained_decoding',
 'mvp_aos_augment-unconstrained_decoding',
 'mvp_aos-constrained_decoding',
 'mvp_aos-unconstrained_decoding',
 'legoabsa_tasktransfer_old-constrained_decoding',
 'legoabsa_tasktransfer_old-unconstrained_decoding']

In [64]:
new_df.columns

Index(['sentence_id', 'task_elements', 'element_order', 'input', 'target',
       'prediction', 'target_list', 'prediction_list', 'metrics', 'f1_score',
       ...
       'f1_score-legoabsa_tasktransfer-constrained_decoding',
       'task_elements-legoabsa_tasktransfer-unconstrained_decoding',
       'element_order-legoabsa_tasktransfer-unconstrained_decoding',
       'input-legoabsa_tasktransfer-unconstrained_decoding',
       'target-legoabsa_tasktransfer-unconstrained_decoding',
       'prediction-legoabsa_tasktransfer-unconstrained_decoding',
       'target_list-legoabsa_tasktransfer-unconstrained_decoding',
       'prediction_list-legoabsa_tasktransfer-unconstrained_decoding',
       'metrics-legoabsa_tasktransfer-unconstrained_decoding',
       'f1_score-legoabsa_tasktransfer-unconstrained_decoding'],
      dtype='object', length=109)

In [72]:
selected_columns = ['sentence_id', 'element_order', 'input', 'target_list', 'prediction_list', 'f1_score', 'prediction_list-mvp_aos_augment-unconstrained_decoding', 'f1_score-mvp_aos_augment-unconstrained_decoding']
for key in df_dict.keys():
	if 'mvp_aos_augment' in key:
		continue
	key = key.replace('_old', '')
	selected_columns.append(f'prediction_list-{key}')
	selected_columns.append(f'f1_score-{key}')
selected_columns

['sentence_id',
 'element_order',
 'input',
 'target_list',
 'prediction_list',
 'f1_score',
 'prediction_list-mvp_aos_augment-unconstrained_decoding',
 'f1_score-mvp_aos_augment-unconstrained_decoding',
 'prediction_list-indolegoabsa_multitask-constrained_decoding',
 'f1_score-indolegoabsa_multitask-constrained_decoding',
 'prediction_list-indolegoabsa_multitask-unconstrained_decoding',
 'f1_score-indolegoabsa_multitask-unconstrained_decoding',
 'prediction_list-gas-constrained_decoding',
 'f1_score-gas-constrained_decoding',
 'prediction_list-gas-unconstrained_decoding',
 'f1_score-gas-unconstrained_decoding',
 'prediction_list-legoabsa_multitask-constrained_decoding',
 'f1_score-legoabsa_multitask-constrained_decoding',
 'prediction_list-legoabsa_multitask-unconstrained_decoding',
 'f1_score-legoabsa_multitask-unconstrained_decoding',
 'prediction_list-mvp_aos-constrained_decoding',
 'f1_score-mvp_aos-constrained_decoding',
 'prediction_list-mvp_aos-unconstrained_decoding',
 'f1_sco

In [73]:
new_df[selected_columns]

,sentence_id,element_order,input,target_list,prediction_list,f1_score,prediction_list-mvp_aos_augment-unconstrained_decoding,f1_score-mvp_aos_augment-unconstrained_decoding,prediction_list-indolegoabsa_multitask-constrained_decoding,f1_score-indolegoabsa_multitask-constrained_decoding,...,prediction_list-legoabsa_multitask-unconstrained_decoding,f1_score-legoabsa_multitask-unconstrained_decoding,prediction_list-mvp_aos-constrained_decoding,f1_score-mvp_aos-constrained_decoding,prediction_list-mvp_aos-unconstrained_decoding,f1_score-mvp_aos-unconstrained_decoding,prediction_list-legoabsa_tasktransfer-constrained_decoding,f1_score-legoabsa_tasktransfer-constrained_decoding,prediction_list-legoabsa_tasktransfer-unconstrained_decoding,f1_score-legoabsa_tasktransfer-unconstrained_decoding
0,3500,aos,pelayanan nya sangat ramah . [A] [O] [S] =>,[A] pelayanan nya [O] sangat ramah [S] positive,[A] pelayanan nya [O] sangat ramah [S] positive,1.000000,[A] pelayanan nya [O] sangat ramah [S] positive,1.000000,<|aspect|> pelayanan nya <|opinion|> sangat ra...,1.000000,...,<|aspect|> pelayanan nya <|opinion|> sangat ra...,1.000000,[A] pelayanan nya [O] sangat ramah [S] positive,1.0,[A] pelayanan nya [O] sangat ramah [S] positive,1.0,<|aspect|> pelayanan nya <|opinion|> sangat ra...,1.0,<|aspect|> pelayanan nya <|opinion|> sangat ra...,1.0
1,3501,aos,sayang wifi tidak bagus harus keluar kamar . [...,[A] wifi [O] tidak bagus harus keluar kamar [S...,[A] wifi [O] tidak bagus [S] negative\n-------...,0.000000,[A] wifi [O] tidak bagus [S] negative\n-------...,0.000000,<|aspect|> wifi <|opinion|> tidak bagus harus ...,1.000000,...,<|aspect|> wifi <|opinion|> tidak bagus harus ...,1.000000,[A] wifi [O] tidak bagus [S] negative\n-------...,0.0,[A] wifi [O] tidak bagus [S] negative\n-------...,0.0,<|aspect|> wifi <|opinion|> tidak bagus harus ...,0.0,<|aspect|> wifi <|opinion|> tidak bagus harus ...,0.0
2,3502,aos,"tulisannya twin bed , tetapi yang ada kamarnya...",[A] kamarnya [O] beda [S] negative,[A] kamarnya [O] twin bed [S] negative\n------...,0.000000,[A] kamarnya [O] twin bed [S] negative\n------...,0.000000,<|aspect|> kamarnya <|opinion|> beda twin bed ...,0.000000,...,<|aspect|> twin bed <|opinion|> ada kamarnya b...,0.000000,"[A] twin bed [O] tulisannya twin bed , tetapi ...",0.0,"[A] twin bed [O] tulisannya twin bed , tetapi ...",0.0,<|aspect|> twin bed <|opinion|> twin bed <|sen...,0.0,<|aspect|> twin bed <|opinion|> twin bed\n----...,0.0
3,3503,aos,"over all baik , hanya sja akan lebih memuaskan...",[A] over all [O] baik [S] positive\n--------\n...,[A] over all [O] baik [S] positive\n--------\n...,0.500000,[A] over all [O] baik [S] positive\n--------\n...,0.500000,<|aspect|> over all <|opinion|> baik <|sentime...,0.500000,...,<|aspect|> over all <|opinion|> baik <|sentime...,0.500000,[A] over all [O] baik [S] positive\n--------\n...,0.5,[A] over all [O] baik [S] positive\n--------\n...,0.5,<|aspect|> over all <|opinion|> baik <|sentime...,0.5,<|aspect|> over all <|opinion|> baik\n--------...,0.0
4,3504,aos,fasilatas sesuia . [A] [O] [S] =>,[A] fasilatas [O] sesuia [S] positive,[A] null [O] sesuia [S] positive\n--------\n[A...,0.000000,[A] fasilitatas [O] sesuia [S] positive,0.000000,<|aspect|> null <|opinion|> sesuia <|sentiment...,0.000000,...,<|aspect|> fasilitatas <|opinion|> sesuia <|se...,0.000000,[A]fasilatas [O] sesuia [S] positive,0.0,[A] fasilatas [O] sesuai [S] positive,0.0,<|aspect|>fasilatas <|opinion|> sesuia,0.0,<|aspect|> fasilatas <|opinion|> sesuai,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1255,2660,aos,ac nya gk dingin [A] [O] [S] =>,[A] ac nya [O] gk dingin [S] negative,[A] ac nya [O] gk dingin [S] negative,1.000000,[A] ac nya [O] gk dingin [S] negative,1.000000,<|aspect|> ac nya <|opinion|> gk dingin <|sent...,1.000000,...,<|aspect|> ac nya <|opinion|> gk dingin <|sent...,1.000000,[A] ac nya [O] gk dingin [S] negative,1.0,[A] ac nya [O] gk dingin [S] negative,1.0,<|asp

In [74]:
new_df[selected_columns].rename({'target_list': 'target_list', 'prediction_list': 'prediction_list-mvp_aos_augment-constrained_decoding', 'f1_score': 'f1_score-mvp_aos_augment-constrained_decoding'}, axis=1).to_csv('notebooks/error_analysis_hoasa_hotel_indo_seed123.csv', index=False)